<a href="https://colab.research.google.com/github/adnanzaidi548/PhD-_Thesis_Data_Codes-/blob/main/MASTER_PAPER01_NEO_AMS_v3_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MASTER PAPER 01 — NEO-AMS v3.0
**Real-source-first, evidence-controlled research workflow (Colab / Jupyter)**  
Paper 01: *Evidence-Controlled Integration of AI, Optimization, and Adaptive Decision Layers for Renewable-Aware Intelligent Manufacturing*.

**Read first:** This notebook is executable research infrastructure, not a promise that all data/scheduling ground truth are present. Forecasting with authentic energy records is **real-data forecasting**. Unless real factory jobs, machine capacities, renewable kWh and electricity tariffs are documented, downstream scheduling is a **constructed manufacturing scenario driven by a real time series**, not measured manufacturing evidence. All numerical outputs retain evidence labels. Do not mislabel estimated/derived/synthetic results as measured.

Related versions: `04_Condensed_1` studies overall architecture; `07_Revised_Condensed_1` develops the formal MIIC contract, six conditions CS,CT,CU,CF,CA,CE. Paper 01 should make only verified distinct empirical claims; text overlap must be separately audited.

## 1. Configuration — inspect and edit before Run all
For Google Drive **shared** folders, create **shortcuts in My Drive** to the five shared folders or set paths to authentic files. A folder ID cannot automatically be treated as a local filesystem path. The `USE_FIXTURE` setting is for smoke testing only and is disabled by default. Choose an authentic source CSV after discovery. The notebook intentionally stops short of a real-data claim when mapping is missing.

In [1]:
import os,sys,json,hashlib,platform,datetime,pathlib,subprocess,zipfile,io,time,math,re
import numpy as np,pandas as pd
from scipy.optimize import milp,LinearConstraint,Bounds
from scipy import stats
from sklearn.metrics import mean_absolute_error,mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
ROOT=pathlib.Path.cwd(); OUT=ROOT/'Paper01_v3_results';OUT.mkdir(exist_ok=True)
REPO='adnanzaidi548/PhD-_Thesis_Data_Codes-'; API='https://api.github.com/repos/'+REPO
DRIVE_FOLDERS={'real':'1Y8Mr7vEkLxkor0sFM6eYKI0GoN4iAvAH','processed':'1qlQbS2IC2C7F8eYileZ5TBVHhNgooj5e','baseline':'1-i1M0bU92aqsg7iUD61m9n1O7cG2taAE','processing_code':'16O19P53CpiN1f95brBtBCynGpkqz_Hmo','processed_bundles':'1gaM4MEG7xKYZuyakWW8hOcLzu_omegba'}
MOUNT_DRIVE = 'google.colab' in sys.modules  # For Colab, we mount explicitly in next cell
DATA_ROOTS=[pathlib.Path('/content/drive/MyDrive'),pathlib.Path('/content'),ROOT]
REAL_CSV=None            # Set absolute path to verified CSV/TSV containing authentic observations
TIMESTAMP_COL=None       # Required e.g. 'timestamp'; do not guess
TARGET_COL=None          # Required e.g. 'load_kW'
TARGET_UNITS=None        # Required e.g. 'kW', 'kWh', 'MW' (state definition)
SOURCE_DESCRIPTION=None # Required: origin/instrument/public series metadata
USE_FIXTURE=False        # Never use fixture for paper claims
FORECAST_HORIZON=1; HOLDOUT_FRACTION=0.25
MAX_DATA_ROWS=120000; MAX_DISCOVERY_FILES=1200
N_SEEDS=8               # Increase e.g. to 30 for publication testing; runtime rises
FAULT_PROBABILITY=0.12
TARIFF_PER_KWH=0.22; GRID_CO2_KG_PER_KWH=0.35; GRID_KWH_LIMIT=50.0
SEED_ROOT=20261011
CONFIG={'forecast_horizon':FORECAST_HORIZON,'n_seeds':N_SEEDS,'fault_probability':FAULT_PROBABILITY,'tariff':TARIFF_PER_KWH,'grid_co2':GRID_CO2_KG_PER_KWH,'grid_limit':GRID_KWH_LIMIT}
print('Python',sys.version.split()[0],'NumPy',np.__version__,'Pandas',pd.__version__,'Output',OUT)

Python 3.13.16 NumPy 2.1.3 Pandas 2.2.3 Output /content/Paper01_v3_results


## 2. Authenticate Google Drive, record repository revision, discover files
The process does **not** automatically download multi-hundred-MB archives. It records folder IDs, accessible filesystem paths, candidate datasets, and file sizes. To access data shared from another Google account, make a shortcut to the shared folder in My Drive or supply a verified local path.

In [ ]:
# Running this cell in Colab requests an interactive Drive authorization.
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive',force_remount=False)
    DRIVE_MOUNTED=True
except ImportError:
    DRIVE_MOUNTED=False;print('Not in Colab; use local datasets or mount Drive externally.')
except Exception as exc:
    DRIVE_MOUNTED=False; print('Drive mount not available:',type(exc).__name__,str(exc)[:220])

In [ ]:
import requests
session=requests.Session();session.headers.update({'Accept':'application/vnd.github+json','User-Agent':'paper01-reproducibility-v3'})
try:
    j=session.get(API+'/git/trees/main?recursive=1',timeout=25);j.raise_for_status();tree=j.json()
    assert not tree.get('truncated',True),'Repository file tree truncated'
    git_sha=tree['sha'];files=[x for x in tree['tree'] if x['type']=='blob']
    pd.DataFrame(files)[['path','sha','size']].to_csv(OUT/'github_inventory.csv',index=False)
    print('GitHub tree',git_sha,'tracked files',len(files))
except Exception as exc:
    git_sha=None;print('GITHUB UNAVAILABLE:',repr(exc),'Manual real CSV still supported.')
(OUT/'drive_source_ids.json').write_text(json.dumps(DRIVE_FOLDERS,indent=2))
for key,id_ in DRIVE_FOLDERS.items():print(key,'https://drive.google.com/drive/folders/'+id_)

In [ ]:
# Discovery excludes notebook checkpoints, limits traversal and reports candidates, not verified units.
rows=[]
for root in DATA_ROOTS:
    if not root.exists():continue
    count=0
    for base,dirs,names in os.walk(root):
        dirs[:]=[d for d in dirs if not d.startswith('.') and d not in ('site-packages','node_modules','.git','outputs','Paper01_v3_results')]
        for filename in names:
            if pathlib.Path(filename).suffix.lower() not in ('.csv','.tsv','.parquet','.zip','.json','.ipynb','.txt'):continue
            p=pathlib.Path(base)/filename
            try: size=p.stat().st_size
            except OSError: continue
            rows.append({'path':str(p),'size_bytes':size,'type':p.suffix.lower(),'root':str(root)})
            count+=1
            if count>=MAX_DISCOVERY_FILES:break
        if count>=MAX_DISCOVERY_FILES:break
inv=pd.DataFrame(rows).drop_duplicates(subset=['path']) if rows else pd.DataFrame(columns=['path','size_bytes','type','root'])
inv.to_csv(OUT/'local_source_inventory.csv',index=False)
print('Discovered',len(inv),'candidate files')
if len(inv):display(inv[inv['type'].isin(['.csv','.tsv','.parquet'])].head(35))

### Source schema preview and archive inventory
This scans a small number of candidate files, showing field names and archive members without automatically extracting large bundles. If nothing appears, place the desired CSV in `/content/drive/MyDrive` or set its absolute path. Inspect the original source documentation before choosing its timestamp, target and units.

In [ ]:
# Inspect first few columns from candidate CSV/TSV files before setting REAL_CSV.
# This inspection is not a confirmation of physical meaning or units.
from pathlib import Path
candidate=inv[inv['type'].isin(['.csv','.tsv'])].sort_values('size_bytes').head(12) if len(inv) else pd.DataFrame()
for _,rec in candidate.iterrows():
    path=Path(rec['path'])
    try:
        preview=pd.read_csv(path,sep='\t' if path.suffix.lower()=='.tsv' else ',',nrows=3)
        print(str(path)[:150], 'columns=', list(preview.columns)[:16])
    except Exception as e:print('Skipped preview',path.name, str(e)[:100])
# Inspect archive member NAMES ONLY. Huge ZIPs are deliberately not unpacked.
for _,rec in inv[inv['type']=='.zip'].sort_values('size_bytes').head(8).iterrows():
    try:
        with zipfile.ZipFile(rec['path']) as z:
            names=z.namelist()
            print('ARCHIVE',rec['path'],'entries',len(names),'first',names[:8])
    except Exception as e:print('Archive not inspectable:',str(e)[:80])


## 3. Explicit real-data mapping and provenance
We **do not** infer units or measurement type merely from a filename. If your CSV is large, only the first `MAX_DATA_ROWS` rows are loaded for this run; its size and SHA-256 are logged. At least 250 valid timestamped observations are required. A fixture can verify notebook functionality, but yields **NO real-data evidence**.

In [ ]:
def digest_file(path,chunk=1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(chunk),b''):h.update(b)
    return h.hexdigest()
if USE_FIXTURE:
    rng=np.random.default_rng(77);k=np.arange(700)
    fixture=pd.DataFrame({'timestamp':pd.date_range('2025-01-01',periods=700,freq='h'), 'energy_kwh':np.maximum(0.2,20+5*np.sin(2*np.pi*k/24)+2*np.cos(2*np.pi*k/168)+rng.normal(0,1.2,len(k)))})
    fixture_path=OUT/'SMOKE_TEST_ONLY_NOT_REAL_DATA.csv';fixture.to_csv(fixture_path,index=False)
    REAL_CSV=str(fixture_path);TIMESTAMP_COL='timestamp';TARGET_COL='energy_kwh';TARGET_UNITS='kWh';SOURCE_DESCRIPTION='PROGRAMMATIC FIXTURE - NOT REAL DATA'
if REAL_CSV is None:
    REAL_READY=False;real_reason='No authentic CSV selected. Set REAL_CSV, TIMESTAMP_COL, TARGET_COL, TARGET_UNITS, SOURCE_DESCRIPTION.'
else:
    real_path=pathlib.Path(REAL_CSV)
    assert real_path.is_file(),'Set REAL_CSV to an existing extracted CSV or TSV, not a folder link.'
    assert all([TIMESTAMP_COL,TARGET_COL,TARGET_UNITS,SOURCE_DESCRIPTION]),'Explicit columns, units and provenance are mandatory.'
    sep='\t' if real_path.suffix.lower()=='.tsv' else ','
    raw=pd.read_csv(real_path,sep=sep,nrows=MAX_DATA_ROWS,low_memory=False)
    assert TIMESTAMP_COL in raw.columns and TARGET_COL in raw.columns, ('Available columns',list(raw.columns))
    ts=pd.to_datetime(raw[TIMESTAMP_COL],errors='coerce',utc=True)
    val=pd.to_numeric(raw[TARGET_COL],errors='coerce')
    series=pd.DataFrame({'timestamp':ts,'y':val}).dropna().sort_values('timestamp').drop_duplicates('timestamp')
    assert len(series)>=250,'Fewer than 250 usable observations';assert series.y.ge(0).all(),'Negative values: verify target and preprocessing'
    assert series.timestamp.is_monotonic_increasing
    gaps=series.timestamp.diff().dropna().dt.total_seconds(); med_sec=float(gaps.median());irregular=float((abs(gaps-med_sec)>0.05*med_sec).mean())
    print('Source mapping:',len(series),'rows, median period sec',med_sec,'irregular fraction',irregular)
    assert med_sec>0,'Invalid time interval'
    if irregular>0.05: print('WARNING: irregular sampling; baseline assumes consecutive time steps. Resample and document before final analysis.')
    source_is_fixture=bool(USE_FIXTURE)
    real_ready=not source_is_fixture
    REAL_READY=real_ready
    real_reason='Verified explicitly mapped external file' if real_ready else 'SMOKE TEST ONLY - no real-data claim'
    provenance={'path':str(real_path),'sha256':digest_file(real_path),'size_bytes':real_path.stat().st_size,'timestamp_col':TIMESTAMP_COL,'target_col':TARGET_COL,'units':TARGET_UNITS,'description':SOURCE_DESCRIPTION,'sampling_median_seconds':med_sec,'irregular_fraction':irregular,'rows_loaded':len(series),'evidence':'REAL_SOURCE_SERIES' if real_ready else 'PROGRAMMATIC_FIXTURE'}
    (OUT/'dataset_provenance.json').write_text(json.dumps(provenance,indent=2))
print('REAL_READY',REAL_READY,real_reason)

## 4. Leakage-controlled chronological forecasting (multiple baselines)
One-step-ahead forecast available at the end of step `t-1`. Prediction features are `y[t-1]`, earlier rolling means, hour/day patterns derived from target timestamps (calendar availability assumed); history-only supervised models trained only on earlier observations. Temporal 75/25 holdout, no random splitting. Metrics: MAE, RMSE, sMAPE (MAPE on near-zero is unreliable). A simple persistence forecast is not automatically 'AI'.

In [ ]:
def build_features(x):
    a=pd.DataFrame({'time':x.timestamp,'y':x.y})
    for lag in (1,2,3,24):a[f'lag_{lag}']=a.y.shift(lag)
    for win in (3,12,24):a[f'roll_{win}']=a.y.shift(1).rolling(win,min_periods=win).mean()
    h=a.time.dt.hour.to_numpy();d=a.time.dt.dayofweek.to_numpy()
    a['hour_sin']=np.sin(2*np.pi*h/24);a['hour_cos']=np.cos(2*np.pi*h/24)
    a['dow_sin']=np.sin(2*np.pi*d/7);a['dow_cos']=np.cos(2*np.pi*d/7)
    return a.dropna().reset_index(drop=True)
if 'series' in globals():
    fm=build_features(series); features=[x for x in fm.columns if x not in ('time','y')]
    split=int(len(fm)*(1-HOLDOUT_FRACTION));assert split>=100 and len(fm)-split>=35
    tr,te=fm.iloc[:split],fm.iloc[split:]
    Xtr,Xte=tr[features].to_numpy(),te[features].to_numpy();ytr,yte=tr.y.to_numpy(),te.y.to_numpy()
    models={'persistence':te.lag_1.to_numpy(),'rolling3':te.roll_3.to_numpy(),'rolling24':te.roll_24.to_numpy()}
    fitters={'ridge':Ridge(alpha=10.0),'hist_gradient_boosting':HistGradientBoostingRegressor(max_iter=130,max_leaf_nodes=15,random_state=SEED_ROOT)}
    for label,model in fitters.items():model.fit(Xtr,ytr);models[label]=np.maximum(0,model.predict(Xte))
    result=[];forecast_rows=[]
    for label,yhat in models.items():
        rmse=float(np.sqrt(mean_squared_error(yte,yhat)));smape=float(np.mean(2*np.abs(yte-yhat)/(np.abs(yte)+np.abs(yhat)+1e-9))*100)
        result.append({'model':label,'mae':float(mean_absolute_error(yte,yhat)),'rmse':rmse,'smape_pct':smape,'evidence':'REAL_SOURCE_FORECAST' if REAL_READY else 'FIXTURE_FORECAST'})
        forecast_rows.extend({'time':str(t),'actual':float(y),'prediction':float(p),'model':label} for t,y,p in zip(te.time,yte,yhat))
    forecast_scores=pd.DataFrame(result).sort_values('mae');forecast_scores.to_csv(OUT/'chronological_forecast_scores.csv',index=False)
    forecast_detail=pd.DataFrame(forecast_rows);forecast_detail.to_csv(OUT/'chronological_forecasts.csv',index=False)
    print('Train',len(tr),'Holdout',len(te),'Strict temporal split');display(forecast_scores)
else: print('FORECAST GATE CLOSED: no real or fixture dataset selected')

## 5. Manufacturing problem definition and unit consistency
Scenario assumptions (NOT sourced factory jobs): 2 machines; 8 indivisible, one-hour jobs, fixed kWh/job; releases/deadlines in whole hours; max one job per machine per hour; renewable availability in kWh/hour; residual grid use is nonnegative and capped; cost = grid-kWh × tariff-per-kWh (currency). Production service = number of completed jobs / number requested. Storage is **absent** (matches `storage_active=false` in original configuration). Forecasted energy source can be used only after establishing physical interpretation. If input data is electric **load**, it is NOT renewable generation. To test scheduling, create an explicitly synthetic renewable series whose daily pattern is scaled using the observed time-series; preserve its constructed label.

In [ ]:
from dataclasses import dataclass
@dataclass(frozen=True)
class Job:
    job_id:int;release:int;deadline:int;energy_kwh:float

def construct_case(seed, observed_profile=None, T=16):
    rng=np.random.default_rng(seed)
    jobs=[]
    for i in range(8):
        release=int(rng.integers(0,8));deadline=int(min(T,release+rng.integers(3,9)))
        jobs.append(Job(i,release,deadline,float(rng.choice([3.,4.,5.,6.]))))
    h=np.arange(T)
    if observed_profile is None:
        shape=0.5+0.5*np.sin(2*np.pi*(h-3)/16)
        renewable=np.maximum(0,5+3.5*shape+rng.normal(0,.6,T))
        source='CONSTRUCTED_SYNTHETIC_RENEWABLE'
    else:
        segment=np.asarray(observed_profile,dtype=float)
        if len(segment)<T:raise ValueError('Need >=16 held-out observations')
        segment=segment[:T]
        # Authentic load/consumption has no renewable measurement semantics: the following is a *synthetic proxy*.
        m=max(float(np.mean(segment)),1e-6)
        renewable=np.clip(5+2.5*(segment/m-1)+rng.normal(0,.25,T),0,11)
        source='SYNTHETIC_RENEWABLE_PROXY_FROM_REAL_SOURCE'
    return jobs,renewable,source

def forecast_at(t, history, horizon):
    # At time t: observed renewable only at indices <t; estimate future with rolling past.
    if len(history)==0: return np.full(horizon,5.0)
    past=np.asarray(history[-min(4,len(history)):]);return np.full(horizon,max(0.,float(np.mean(past))))

def validate_action(ids, t, jobs, done, machine_capacity=2):
    ids=list(ids)
    if len(ids)!=len(set(ids)):return False,'duplicate_job'
    if len(ids)>machine_capacity:return False,'machine_overload'
    lookup={j.job_id:j for j in jobs}
    if any(i not in lookup for i in ids):return False,'unknown_job'
    if any(i in done for i in ids):return False,'duplicate_execution'
    if any(not (lookup[i].release<=t<lookup[i].deadline) for i in ids):return False,'release_deadline'
    if sum(lookup[i].energy_kwh for i in ids)>GRID_KWH_LIMIT+11:return False,'energy_capacity'
    return True,'ok'

def policy_action(jobs,done,t,renew_forecast,method):
    available=[j for j in jobs if j.job_id not in done and j.release<=t<j.deadline]
    if not available:return []
    if method=='edf':return [j.job_id for j in sorted(available,key=lambda j:(j.deadline,j.job_id))[:2]]
    if method=='energy':
        return [j.job_id for j in sorted(available,key=lambda j:(j.deadline, j.energy_kwh if renew_forecast<7 else -j.energy_kwh))[:2]]
    if method!='online_milp':raise ValueError(method)
    # Binary x[j,slot] for future slots; each job scheduled <=1, machine capacity <=2 per slot;
    # deadlines/release encoded in candidate feasibility. Missed jobs incur fixed penalty 100.
    horizon=16; candidates=[(j,s) for j in available for s in range(t,min(j.deadline,horizon))]
    if not candidates:return []
    n=len(candidates); c=np.array([max(0,j.energy_kwh-renew_forecast)*TARIFF_PER_KWH+0.001*(s-t)-100.0 for j,s in candidates])
    A=[];lo=[];hi=[]
    for j in available:
        A.append(np.array([float(ja.job_id==j.job_id) for ja,s in candidates]));lo.append(0);hi.append(1)
    for s in range(t,horizon):
        A.append(np.array([float(v==s) for j,v in candidates]));lo.append(0);hi.append(2)
    sol=milp(c,integrality=np.ones(n),bounds=Bounds(np.zeros(n),np.ones(n)),constraints=LinearConstraint(np.vstack(A),lo,hi),options={'time_limit':4.0})
    if sol.x is None:return policy_action(jobs,done,t,renew_forecast,'edf')
    return [j.job_id for (j,s),x in zip(candidates,sol.x) if s==t and x>0.5]

## 6. Matched online decision and contract ablations
For *every* seed and condition we evaluate identical jobs/renewable trajectories, with EDF, energy heuristic, online MILP, and the **same MILP policy** with admission disabled/enabled. Controlled fault injection targets invalid action IDs; without admission the physical execution layer refuses unsafe commands (no fictional machine execution). With admission, the candidate is repaired to a safe earliest-deadline action. This proves only performance of this specified toy interface, not six-condition MIIC completeness or industrial superiority. Track attempted violations, rejected actions, repair latency, achieved service, cost, emissions. The online MILP is non-anticipatory: it never sees future actual renewable generation.

In [ ]:
def run_episode(seed,method='online_milp',contract=False,fault=False,observed_profile=None):
    jobs,ren,source=construct_case(seed,observed_profile)
    rng=np.random.default_rng(seed+90909)
    done=set();cost=0.;emissions=0.;attempts=0;rejections=0;repairs=0;time_ms=0.;grid_viols=0;logs=[]
    for t in range(len(ren)):
        prediction=float(forecast_at(t,ren[:t],1)[0])
        action=policy_action(jobs,done,t,prediction,method)
        original=list(action)
        fault_applied=False
        if fault and len(action)>0 and rng.random()<FAULT_PROBABILITY:
            action=list(action)+[int(action[0])];fault_applied=True
        valid,why=validate_action(action,t,jobs,done)
        if not valid:
            attempts+=1
            if contract:
                start=time.perf_counter()
                # Repair uses exactly the same deterministic safe fallback for every method.
                action=policy_action(jobs,done,t,prediction,'edf')
                ok2,_=validate_action(action,t,jobs,done)
                assert ok2
                time_ms+=(time.perf_counter()-start)*1000;repairs+=1
            else:
                # Independent physical plant blocks invalid action. No unsafe execution claims.
                action=[];rejections+=1
        energy=sum(j.energy_kwh for j in jobs if j.job_id in action)
        grid=max(0.,energy-ren[t]);grid_viols+=int(grid>GRID_KWH_LIMIT+1e-9)
        for j in action:done.add(j)
        cost+=TARIFF_PER_KWH*grid;emissions+=GRID_CO2_KG_PER_KWH*grid
        logs.append({'seed':seed,'t':t,'method':method,'contract':contract,'fault':fault,'actual_renewable_kwh':float(ren[t]),'past_only_forecast_kwh':prediction,'candidate_action':json.dumps(original),'applied_action':json.dumps(action),'injected_fault':fault_applied,'invalid_candidate':not valid,'grid_kwh':grid,'energy_kwh':energy})
    return {'seed':seed,'method':method,'contract':contract,'fault':fault,'cost':cost,'emissions_kg':emissions,'service':len(done)/len(jobs),'attempted_invalid':attempts,'physical_rejections':rejections,'repairs':repairs,'repair_latency_ms_total':time_ms,'grid_capacity_violations':grid_viols,'evidence':source},logs
all_res=[];all_logs=[]
for seed in range(SEED_ROOT,SEED_ROOT+N_SEEDS):
    obs=None
    if 'forecast_detail' in globals() and REAL_READY:
        sub=forecast_detail[forecast_detail.model=='persistence'].actual.to_numpy()
        if len(sub)>=16:obs=sub[(seed-SEED_ROOT)%max(1,(len(sub)-16)+1):][:16]
    for condition in (False,True):
        for method,on in [('edf',False),('energy',False),('online_milp',False),('online_milp',True)]:
            if on and method!='online_milp':continue
            res,logs=run_episode(seed,method,on,condition,obs)
            all_res.append(res);all_logs.extend(logs)
schedule_runs=pd.DataFrame(all_res);schedule_logs=pd.DataFrame(all_logs)
schedule_runs.to_csv(OUT/'matched_online_scheduling_runs.csv',index=False)
schedule_logs.to_csv(OUT/'action_trace.csv',index=False)
assert schedule_runs.service.between(0,1).all();assert (schedule_runs.grid_capacity_violations==0).all()
assert len(schedule_runs)==N_SEEDS*2*4
print('Episodes:',len(schedule_runs),'event records:',len(schedule_logs))
display(schedule_runs.groupby(['fault','method','contract']).agg(n=('seed','size'),cost_mean=('cost','mean'),service_mean=('service','mean'),invalid_mean=('attempted_invalid','mean'),reject_mean=('physical_rejections','mean'),repair_mean=('repairs','mean')).reset_index())

## 7. Paired inference, ablation validity and decision-relevant value
Comparisons pair identical seeds and fault regimes. Bootstrap confidence intervals quantify mean paired differences, **not** established causality in other operating conditions. Outcome sign: Δ = contract-on − contract-off. An isolated contract test must compare the same base optimizer and fault stream. All results include unfavorable or null outcomes.

In [ ]:
def paired_ci(x,seed=71,iterations=2500):
    x=np.asarray(x,float);rng=np.random.default_rng(seed);n=len(x)
    if n<3:return (float(np.mean(x)),float('nan'),float('nan'))
    draws=rng.integers(0,n,size=(iterations,n));dist=x[draws].mean(axis=1)
    return (float(x.mean()),*map(float,np.quantile(dist,[.025,.975])))
comps=[]
for faulty in (False,True):
    part=schedule_runs[schedule_runs.fault==faulty]
    base=part[(part.method=='online_milp')&(~part.contract)].set_index('seed')
    on=part[(part.method=='online_milp')&(part.contract)].set_index('seed')
    for metric in ('cost','service','emissions_kg','physical_rejections','repairs'):
        delta=(on[metric]-base[metric]).sort_index()
        mean,low,high=paired_ci(delta.to_numpy())
        comps.append({'fault':faulty,'comparison':'contract_on_minus_off','metric':metric,'pairs':len(delta),'mean_difference':mean,'ci95_low':low,'ci95_high':high,'evidence':'CONSTRUCTED_MANUFACTURING_SCENARIO'})
    edf=part[part.method=='edf'].set_index('seed')
    for metric in ('cost','service'):
        delta=(base[metric]-edf[metric]).sort_index()
        mean,low,high=paired_ci(delta.to_numpy())
        comps.append({'fault':faulty,'comparison':'online_milp_minus_edf','metric':metric,'pairs':len(delta),'mean_difference':mean,'ci95_low':low,'ci95_high':high,'evidence':'CONSTRUCTED_MANUFACTURING_SCENARIO'})
paired_results=pd.DataFrame(comps);paired_results.to_csv(OUT/'paired_statistics.csv',index=False)
display(paired_results)

## 8. Integrity controls, plots, release gates, and single downloadable evidence ZIP
Do not treat forecast data as factory data. All scenario-driven results carry `CONSTRUCTED_MANUFACTURING_SCENARIO` or `SYNTHETIC_RENEWABLE_PROXY_FROM_REAL_SOURCE`. Publication release gates are deliberately independent of code completion.

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(9,4));sel=schedule_runs[(~schedule_runs.fault)].groupby(['method','contract'])['cost'].mean()
sel.plot(kind='bar',ax=ax);ax.set_ylabel('Simulated scheduling cost (arbitrary currency)');ax.set_title('Constructed scenario only — not measured industrial evidence');fig.tight_layout();fig.savefig(OUT/'constructed_scenario_cost.png',dpi=170);plt.close(fig)
result_gate={
 'executed_with_authentic_source':bool(REAL_READY),
 'forecast_chronological_holdout':bool(REAL_READY and 'forecast_scores' in globals()),
 'observed_manufacturing_jobs':False,
 'authentic_renewable_gen_and_tariff':False,
 'online_comparator_executed':bool(len(schedule_runs)>0),
 'contract_ablation_executed':bool(len(paired_results)>0),
 'miic_six_contracts_validated':False,
 'original_full_pipeline_reproduced':False,
 'related_paper_distinction_reviewed':False,
 'all_references_doi_verified':False,
 'journal_ready':False}
(OUT/'release_gates.json').write_text(json.dumps(result_gate,indent=2))
trace={'timestamp_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'python':platform.python_version(),'repo':REPO,'tree_sha':git_sha,'config':CONFIG,'data_file':str(REAL_CSV),'dataset_evidence':real_reason,'relevant_manuscripts':['01_Lengthy_1','04_Condensed_1','07_Revised_Condensed_1'],'known_source_issues':['service 1.0 versus 0.774186 original optimisation','full-system mixed latency definitions'],'claims_policy':'No generated/derived/synthetic scheduling figures represented as measured plant outcomes'}
(OUT/'execution_manifest.json').write_text(json.dumps(trace,indent=2,default=str))
checks=[]
for p in sorted(OUT.rglob('*')):
    if p.is_file() and p.name!='file_sha256_manifest.csv':checks.append({'path':str(p.relative_to(OUT)),'sha256':digest_file(p),'size':p.stat().st_size})
pd.DataFrame(checks).to_csv(OUT/'file_sha256_manifest.csv',index=False)
zip_dest=ROOT/'Paper01_v3_EVIDENCE_EXPORT.zip'
with zipfile.ZipFile(zip_dest,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file():z.write(p,arcname=str(p.relative_to(OUT)))
print('EVIDENCE ZIP',zip_dest, 'files',len(checks),'size',zip_dest.stat().st_size)
print('PUBLICATION READY:',result_gate['journal_ready']);display(pd.DataFrame([{'gate':k,'passed':v} for k,v in result_gate.items()]))
try:
    from google.colab import files as colabfiles
    colabfiles.download(str(zip_dest))
except ImportError:pass

## Required follow-up after execution
1. Upload `Paper01_v3_EVIDENCE_EXPORT.zip` and the executed `.ipynb` to ChatGPT.
2. To obtain authentic forecasting results, explicitly set `REAL_CSV`, `TIMESTAMP_COL`, `TARGET_COL`, `TARGET_UNITS`, `SOURCE_DESCRIPTION` after Drive discovery. A source that measures consumption cannot be relabelled solar generation.
3. To claim manufacturing outcomes, supply actual plant jobs, machine ratings, dispatchable generation, energy tariffs and timestamps. Until then all job-scheduling and cost/emissions outcomes remain scenario-generated.
4. For reliable publication estimates, increase `N_SEEDS`, hold fixed assumptions, run stress levels, and audit solver convergence; avoid reporting only significant results.
5. Resolve related-manuscript overlap and bibliography separately; a notebook cannot certify originality or publisher compliance.